# 1. Load silver tables

In [0]:
table_names = ['orders', 'customers', 'order_items', 'products']

In [0]:
silver_tables = {}

for table_name in table_names:
    silver_tables[table_name] = spark.table('olist.silver.' + table_name)

# 2. Sales per month and year

In [0]:
df_order_items_orders = (
    silver_tables['orders']
    .join(silver_tables['order_items'], 
          on='order_id', 
          how='inner'
          )
)

In [0]:
df_order_items_orders.display()

In [0]:
from pyspark.sql import functions as sf

df_order_items_orders = (
    df_order_items_orders.
    withColumn('order_purchase_year_month', 
               sf.date_format("order_purchase_timestamp", "yyyy-MM")
               )
)

df_order_items_orders.display()

In [0]:
df_gold_revenue_per_year_month = (
    df_order_items_orders
    .filter(sf.col('order_status') == "delivered")
    .groupBy('order_purchase_year_month')
    .agg(
        sf.count_distinct('order_id').alias('order_count'),
        sf.count('order_item_id').alias('item_count'),
        sf.round(sf.sum('price'), 2).alias('revenue'),
        sf.round(sf.sum('freight_value'),2).alias('total_freight_value')
    )
    .sort("order_purchase_year_month")
)

df_gold_revenue_per_year_month.display()

In [0]:
# verify if month year is unique
(
    df_gold_revenue_per_year_month
    .groupby("order_purchase_year_month")
    .count()
    .filter(sf.col('count') > 1)
    .display()
)

In [0]:
# verify if revenue isn't null
(
    df_gold_revenue_per_year_month
    .filter(sf.col("revenue").isNull())
    .display()

)

# 3. Delivery quality by customer state

What's the average delivery delay by state?

In [0]:
df_gold_delay_by_state = (
    silver_tables['orders']
    .groupBy('customer_state')
    .agg(
        sf.count_distinct(
            sf.when(
                sf.col("is_late").isNotNull(),
                sf.col("order_id")
            
        )).alias('delivered_orders'),

        sf.count_distinct(
            sf.when(
                sf.col("is_late") == True,
                sf.col("order_id")
            )
        ).alias('late_orders'),

        sf.round(
            sf.mean(
                sf.when(
                    sf.col("is_late") == True,
                    sf.col("delivery_delay_days")
                    )
                ), 
            2).alias('avg_late_delay_days')
            
    )

    .withColumn('late_delivery_rate_percentage', 
                sf.round(
                    sf.col('late_orders') / sf.col('delivered_orders') * 100, 
                    2 ))
)

df_gold_delay_by_state.display()

In [0]:
# check if customer state is unique
(
    df_gold_delay_by_state
    .groupBy('customer_state')
    .count()
    .filter(sf.col('count') > 1)
    .display()
)

In [0]:
# check if late_orders <= delivered_orders
# if no rows are displayed, then the check has passed

(
    df_gold_delay_by_state
    .filter(
        sf.col('late_orders') > sf.col('delivered_orders')
    )
    .display()
)

In [0]:
# check if rate is between 0 and 100
(
    df_gold_delay_by_state
    .filter(
        (sf.col('late_delivery_rate_percentage') > 100)
        | (sf.col('late_delivery_rate_percentage') < 0)
    )
    .display()
)

# 4. Revenue by product category

In [0]:
# join order_items with products
df_order_items_products = silver_tables['order_items'].join(silver_tables['products'], on='product_id', how='inner')

df_order_items_products.display()

In [0]:
df_order_items_products_orders = (
    df_order_items_products
    .join(silver_tables['orders'].select('order_id', 'order_status'), on='order_id', how='inner')
)


In [0]:
df_gold_revenue_by_product_category = (
    df_order_items_products_orders
    .filter(sf.col('order_status') == 'delivered')
    .withColumn('product_category_name',
        sf.coalesce(
            sf.col('product_category_name'),
            sf.lit("unknown")
        )
    )
    .groupBy('product_category_name')
    .agg(
        sf.count_distinct('order_id').alias('order_count'),
        sf.count('order_item_id').alias('item_count'),
        sf.round(sf.sum('price'), 2).alias('revenue'),
        sf.round(sf.mean('price'), 2).alias('avg_item_price')
    )

)

df_gold_revenue_by_product_category.display()

In [0]:
# check if product_category_name is unique
# if no rows are returned then the test was passed
(
    df_gold_revenue_by_product_category
    .groupBy('product_category_name')
    .count()
    .filter(sf.col('count') > 1)
    .display()
)

# 5. Save gold tables

In [0]:
gold_tables = {
    "monthly_sales": df_gold_revenue_per_year_month,
    "delivery_performance_by_state": df_gold_delay_by_state,
    "product_category_performance": df_gold_revenue_by_product_category
}

for name, df in gold_tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"olist.gold.{name}")
    )

In [0]:
# read back the tables
for name in gold_tables:
    df = spark.table(f"olist.gold.{name}")
    print(name, df.count())